In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,average_precision_score, roc_curve, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

import joblib


In [2]:
# 1. Establish a reproducible directory structure
os.makedirs("eda_plots", exist_ok=True)
os.makedirs("dataset", exist_ok=True)

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'


In [3]:
# 2. Ingest data with robust missing value handling
train_path = "dataset/raw_data/aps_failure_training_set.csv"
test_path = "dataset/raw_data/aps_failure_test_set.csv"

# Professional reasoning: skiprows=20 skips UCI metadata documentation header text.
# na_values="na" forces Pandas to recognize text-based missing strings as true NaN objects instantly.
train_df = pd.read_csv(train_path, skiprows=20, na_values="na")
test_df = pd.read_csv(test_path, skiprows=20, na_values="na")


In [4]:
# 3. Seperate Features (X) and Target (y) with explicit binary mapping
y_train = train_df['class'].map({'neg': 0, 'pos': 1}).values
X_train_raw = train_df.drop(columns=['class'])

y_test = test_df['class'].map({'neg': 0, 'pos': 1}).values
X_test_raw = test_df.drop(columns=['class'])

print(f"Training Data Loaded Successfully: {X_train_raw.shape}")
print(f"Testing Data Loaded Successfully: {X_test_raw.shape}")

Training Data Loaded Successfully: (60000, 170)
Testing Data Loaded Successfully: (16000, 170)


In [5]:
# 4. Class Imbalance Audit

class_counts = pd.Series(y_train).value_counts()
neg_count = class_counts[0]
pos_count = class_counts[1]
imbalance_ratio = neg_count / pos_count

print(f"-----Class Imbalance Audit-----")
print(f"Normal instances(neg): {neg_count:,}")
print(f"Failure instances(pos): {pos_count:,}")
print(f"Calculated Imbalance Ratio: {imbalance_ratio:.0f}:1")

-----Class Imbalance Audit-----
Normal instances(neg): 59,000
Failure instances(pos):, 1,000
Calculated Imbalance Ratio: 59:1


In [6]:
# 5. EDA PLOT 1: Class Imbalance Ratio
plt.figure(figsize=(6, 4))
sns.countplot(x=train_df['class'], hue=train_df['class'], palette=['#2b5c8f', '#d9534f'], legend=False)
plt.title(f"Class Distribution: Normal vs APS Failure ({imbalance_ratio:.0f}:1 Ratio)")
plt.xlabel("Operational Status")
plt.ylabel("Truck Count")
plt.savefig("eda_plots/01_class_imbalance.png", dpi=300, bbox_inches='tight')
plt.close()

In [7]:
# 6. Domain-Specific Feature Engineering

def enrich_features(df):
    df_fe = df.copy()
    # 1. Row-wise missingness signature (ECU dropout indicator)
    df_fe['fe_missing_count'] = df.isna().sum(axis=1)
    df_fe['fe_missing_ratio'] = df_fe['fe_missing_count'] / df.shape[1]

    # 2. Operational Histogram Aggregations(7 families, 10 bins each)
    hist_prefixes = ['ag_', 'ay_', 'az_', 'ba_', 'cn_', 'cs_', 'ee_']
    for prefix in hist_prefixes:
        cols = [c for c in df.columns if c.startswith(prefix)]
        if cols:
            df_fe[f'fe_{prefix}sum'] = df[cols].sum(axis=1, skipna=True)
            df_fe[f'fe_{prefix}max'] = df[cols].max(axis=1, skipna=True)
    return df_fe

X_train_fe = enrich_features(X_train_raw)
X_test_fe = enrich_features(X_test_raw)

print(f"Features after domain enrichment: {X_train_fe.shape[1]} (Added 16 engineered features)")

Features after domain enrichment: 186 (Added 16 engineered features)


In [8]:
# 7. Data Cleaning and Missingness Pruning

keep_cols = X_train_fe.columns[X_train_fe.isna().mean() <= 0.70].tolist()
X_train_clean = X_train_fe[keep_cols]
X_test_clean = X_test_fe[keep_cols]

print(f"Pruned {X_train_fe.shape[1] - len(keep_cols)} columns with >70% missingness.")

Pruned 7 columns with >70% missingness.


In [9]:
# 8. EDA PLOT 2: Missingness Profile Across Sensors

missing_pct = X_train_raw.isna().mean() * 100
plt.figure(figsize=(8, 4))
plt.hist(missing_pct, bins=30, color='#4682b4', edgecolor='black')
plt.axvline(70, color='crimson', linestyle='--', linewidth=2, label='70% Drop Threshold')
plt.title("Missing Value Percentage Profile Across 170 Sensors")
plt.xlabel("Missing Percentage (%)")
plt.ylabel("Number of Sensors")
plt.legend()
plt.savefig("eda_plots/02_missingness_profile.png", dpi=300, bbox_inches='tight')
plt.close()

In [10]:
# 9. Missing Value Imputation (Median Strategy)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_clean)
X_test_imp = imputer.transform(X_test_clean)

# Remove Zero-Variance Constant Features
var_selector = VarianceThreshold(threshold=0.0)
X_train_var = var_selector.fit_transform(X_train_imp)
X_test_var = var_selector.transform(X_test_imp)
surviving_features = np.array(keep_cols)[var_selector.get_support()]

print(f"Features after zero-variance removal: {X_train_var.shape[1]}")

Features after zero-variance removal: 178


In [11]:
# 10. Feature Scaling (RobustScaler for Outlier Resilience)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_var)
X_test_scaled = scaler.transform(X_test_var)

# Clean any residual numerical anomalies (inf / NaN values generated by scaling bounds)
X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

In [12]:
# 11. Dimensionality Reduction(PCA)

pca = PCA(n_components=50, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

explained_variance_sum = pca.explained_variance_ratio_.sum() * 100

print(f"----PCA Dimensionality Reduction Complete----")
print(f"Original Scaled Features: {X_train_scaled.shape[1]}")
print(f"PCA Compressed Features: {X_train_pca.shape[1]}")
print(f"Total Variance Retained (50 Components): {explained_variance_sum:.2f}%")

----PCA Dimensionality Reduction Complete----
Original Scaled Features: 178
PCA Compressed Features: 50
Total Variance Retained (50 Components): 100.00%


In [13]:
# 12. EDA PLOT 3: Sensor Distribution Comparison
plt.figure(figsize=(7, 4))
sns.boxplot(x=y_train, hue= y_train, y=X_train_raw['aa_000'], palette=["#2b5c8f", "#d9534f"], legend=False)
plt.xticks([0, 1], ['Normal (neg)', 'Failure (pos)'])
plt.title("Distribution of Sensor aa_000 for Normal vs Failed Trucks")
plt.xlabel("True Operational Class")
plt.ylabel("Sensor aa_000 Reading")
plt.savefig("eda_plots/03_sensor_distributions.png", dpi=300, bbox_inches='tight')
plt.close()


In [14]:
# 13. EDA PLOT 4: Correlation Heatmap (Multicollinearity Check)
plt.figure(figsize=(8, 6))
corr_matrix = pd.DataFrame(X_train_scaled[:, :15]).corr()
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, cbar=True)
plt.title("Feature Correlation Heatmap (Top 15 Scaled Features)")
plt.savefig("eda_plots/04_correlation_heatmap.png", dpi=300, bbox_inches='tight')
plt.close()

In [15]:
# 14. EDA PLOT 5: PCA Scree Plot (Cumulative Variance)
plt.figure(figsize=(7, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_) * 100, color='#2b5c8f', lw=2.5)
plt.axhline(95, color='crimson', linestyle='--', linewidth=2, label='95% Variance Threshold')
plt.title(f"PCA Cumulative Explained Variance ({X_train_pca.shape[1]} Components)")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Variance Explained (%)")
plt.legend(loc='lower right')
plt.savefig("eda_plots/05_pca_scree_plot.png", dpi=300, bbox_inches='tight')
plt.close()

In [16]:
# 15. Manual Pipeline Cross-Check
assert np.isnan(X_train_scaled).sum() == 0, "ERROR: NaN values detected in scaled training matrix!"
assert np.isinf(X_train_scaled).sum() == 0, "ERROR: Inf values detected in scaled training matrix!"
assert X_train_pca.shape[1] == 50,         "ERROR: PCA component count mismatch!"
assert explained_variance_sum > 80.0,       "ERROR: Retained variance is below acceptable threshold!"

print("ALL MANUAL SANITY CHECKS PASSED SUCCESSFULLY. Data pipeline is robust and ready for model training.")

ALL MANUAL SANITY CHECKS PASSED SUCCESSFULLY. Data pipeline is robust and ready for model training.


In [17]:
# 16. Class Imbalance Mitigation (SMOTE + Random Under-Sampling)

# SMOTE on PCA space (20% minority ratio to avoid memory explosion)
smote = SMOTE(sampling_strategy=0.20, random_state=42)
X_train_pca_smote, y_train_pca_smote = smote.fit_resample(X_train_pca, y_train)

# Random Undersampling on PCA space (Balanced 2:1 ratio for fast distance learners)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_rus_pca, y_train_rus_pca = rus.fit_resample(X_train_pca, y_train)

print(f"Original Train Shape: {X_train_pca.shape} | Failures: {y_train.sum()}")
print(f"SMOTE Train Shape: {X_train_pca_smote.shape} | Failures: {y_train_pca_smote.sum()}")
print(f"Undersampled Train Shape: {X_train_rus_pca.shape} | Failures: {y_train_rus_pca.sum()}")

Original Train Shape: (60000, 50) | Failures: 1000
SMOTE Train Shape: (70800, 50) | Failures: 11800
Undersampled Train Shape: (3000, 50) | Failures: 1000


In [18]:
# 17. Exhaustive Supervised Model Benchmarking & Cost Optimization

def scania_cost(y_true, y_pred):
    """Official UCI Scania Cost Function: $10 * FP + $500 * FN"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return (10 * fp) + (500 * fn), fp, fn, tp

def evaluate_with_optimal_threshold(name, model, X_tr, y_tr, X_te, y_te):
    """Fits model, finds decision threshold minimizing Scania Cost on validation split, evaluates on Test."""
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42
    )
    model.fit(X_fit, y_fit)

    val_probs  = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_te)[:, 1]

    # Sweep thresholds on Validation set to minimize Official Scania Cost
    thresholds = np.linspace(0.01, 0.90, 90)
    best_t, min_val_cost = 0.50, float('inf')
    for t in thresholds:
        c, _, _, _ = scania_cost(y_val, (val_probs >= t).astype(int))
        if c < min_val_cost:
            min_val_cost, best_t = c, t

    # Apply optimal threshold to unseen Test Set
    y_test_pred = (test_probs >= best_t).astype(int)
    opt_cost, fp, fn, tp = scania_cost(y_te, y_test_pred)

    return {
        "Model": name,
        "Best_Thresh": round(best_t, 3),
        "ROC_AUC": round(roc_auc_score(y_te, test_probs), 4),
        "PR_AUC": round(average_precision_score(y_te, test_probs), 4),
        "Recall": round(recall_score(y_te, y_test_pred), 4),
        "Precision": round(precision_score(y_te, y_test_pred, zero_division=0), 4),
        "F1_Score": round(f1_score(y_te, y_test_pred), 4),
        "FP ($10)": fp,
        "FN ($500)": fn,
        "Optimal Cost ($)": opt_cost
    }, test_probs

# Benchmark Key Classifiers
# Updated models list with increased max_iter for Logistic Regression
models_to_test = [
    ("Logistic Regression (Balanced + PCA)", LogisticRegression(class_weight='balanced', max_iter=5000, tol=1e-3, solver='saga', random_state=42), X_train_pca, y_train, X_test_pca),
    ("Random Forest (Balanced Subsample)", RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=42, n_jobs=-1), X_train_scaled, y_train, X_test_scaled),
    ("HistGradientBoosting (Balanced)", HistGradientBoostingClassifier(max_iter=100, class_weight='balanced', random_state=42), X_train_scaled, y_train, X_test_scaled)
]

print("---- RUNNING MODEL BENCHMARK & COST OPTIMIZATION----")
results = []
for name, clf, X_tr, y_tr, X_te in models_to_test:
    print(f"Training and optimizing: {name}...")
    res, t_probs = evaluate_with_optimal_threshold(name, clf, X_tr, y_tr, X_te, y_test)
    results.append(res)
    if "Random Forest" in name:
        best_test_probs = t_probs

results_df = pd.DataFrame(results).sort_values(by="Optimal Cost ($)")
print("----BENCHMARK RESULTS (SORTED BY LOWEST COST)----")
print(results_df.to_string(index=False))


---- RUNNING MODEL BENCHMARK & COST OPTIMIZATION----
Training and optimizing: Logistic Regression (Balanced + PCA)...
Training and optimizing: Random Forest (Balanced Subsample)...
Training and optimizing: HistGradientBoosting (Balanced)...
----BENCHMARK RESULTS (SORTED BY LOWEST COST)----
                               Model  Best_Thresh  ROC_AUC  PR_AUC  Recall  Precision  F1_Score  FP ($10)  FN ($500)  Optimal Cost ($)
  Random Forest (Balanced Subsample)         0.06   0.9906  0.8776  0.9573     0.5241    0.6774       326         16             11260
     HistGradientBoosting (Balanced)         0.29   0.9930  0.9006  0.9493     0.5258    0.6768       321         19             12710
Logistic Regression (Balanced + PCA)         0.54   0.9112  0.4743  0.6187     0.2742    0.3800       614        143             77640


In [19]:
# 18. EDA PLOT 6: Decision Tree Bias-Variance Trade-off

depths = [2, 4, 6, 8, 10, 12, 16, 20, 25]
train_f1s, val_f1s = [], []
X_tr_bv, X_val_bv, y_tr_bv, y_val_bv = train_test_split(
    X_train_scaled, y_train, test_size=0.2, stratify=y_train, random_state=42
)
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, class_weight='balanced', random_state=42)
    dt.fit(X_tr_bv, y_tr_bv)
    train_f1s.append(f1_score(y_tr_bv, dt.predict(X_tr_bv)))
    val_f1s.append(f1_score(y_val_bv, dt.predict(X_val_bv)))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_f1s, marker='o', color='#2b5c8f', label='Training F1 (Low Bias)')
plt.plot(depths, val_f1s, marker='s', color='#d9534f', label='Validation F1 (Generalization)')
plt.title("Bias-Variance & Complexity Trade-off (Decision Tree Max Depth)")
plt.xlabel("Max Tree Depth")
plt.ylabel("Minority Class F1-Score")
plt.legend()
plt.savefig("eda_plots/06_bias_variance_tradeoff.png", dpi=300, bbox_inches='tight')
plt.close()

In [20]:
# 19. EDA PLOT 7: Cost vs Threshold Curve

ts = np.linspace(0.01, 0.90, 90)
costs = [scania_cost(y_test, (best_test_probs >= t).astype(int))[0] for t in ts]
plt.figure(figsize=(7, 4))
plt.plot(ts, costs, lw=2.5, color='crimson')
plt.xlabel("Decision Probability Threshold (t)")
plt.ylabel("Total Scania Maintenance Cost ($)")
plt.title("Cost-Sensitive Threshold Optimization Curve (Random Forest)")
plt.savefig("eda_plots/07_cost_threshold_curve.png", dpi=300, bbox_inches='tight')
plt.close()

In [21]:
# 20. EDA PLOT 8: ROC Curve

fpr, tpr, _ = roc_curve(y_test, best_test_probs)
plt.figure(figsize=(7, 4))
plt.plot(fpr, tpr, color='#2b5c8f', lw=2, label='Random Forest (AUC = 0.9906)')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc='lower right')
plt.savefig("eda_plots/08_roc_and_pr_curves.png", dpi=300, bbox_inches='tight')
plt.close()

In [22]:
# 21. Unsupervised Failure Clustering

# Isolate positive failure cases in PCA space (top 15 components)
X_failures_pca = X_train_pca[y_train == 1][:, :15]

print("\n--- RUNNING UNSUPERVISED CLUSTERING ON FAILED TRUCKS ---")
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_failures_pca)
    sil = silhouette_score(X_failures_pca, km.labels_)
    dbi = davies_bouldin_score(X_failures_pca, km.labels_)
    print(f"K-Means (k={k}): Silhouette Score = {sil:.4f} | Davies-Bouldin Index = {dbi:.4f}")


--- RUNNING UNSUPERVISED CLUSTERING ON FAILED TRUCKS ---
K-Means (k=2): Silhouette Score = 0.9776 | Davies-Bouldin Index = 0.2195
K-Means (k=3): Silhouette Score = 0.9683 | Davies-Bouldin Index = 0.4130
K-Means (k=4): Silhouette Score = 0.9669 | Davies-Bouldin Index = 0.3520


In [23]:
# 22. EDA PLOT 9: Hierarchical Dendrogram

plt.figure(figsize=(8, 4))
Z = linkage(X_failures_pca[:250], method='ward')
dendrogram(Z, truncate_mode='lastp', p=12)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage on Failed Trucks)")
plt.xlabel("Cluster Size / Index")
plt.ylabel("Ward Euclidean Distance")
plt.savefig("eda_plots/09_failure_dendrogram.png", dpi=300, bbox_inches='tight')
plt.close()

In [24]:
# 23. EDA PLOT 10: 2D Cluster Scatter Plot

km_final = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_failures_pca)
plt.figure(figsize=(6, 4))
plt.scatter(X_failures_pca[:, 0], X_failures_pca[:, 1], c=km_final.labels_, cmap='viridis', alpha=0.7)
plt.title("Failed Trucks Clustered in 2D PCA Subspace (K-Means k=3)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.savefig("eda_plots/10_clusters_2d_pca.png", dpi=300, bbox_inches='tight')
plt.close()

In [25]:
# 24. Train Final Production Random Forest & Save All Artifacts

print("Fitting and saving production model and pipeline artifacts...")

# 1. Fit the final production Random Forest model
final_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
final_rf.fit(X_train_scaled, y_train)

# 2. Serialize and dump the model, scaler, and pca transformer using joblib
joblib.dump(final_rf, "scania_rf_best.pkl")
joblib.dump(scaler,   "scania_scaler.pkl") # Matches 'scaler' object from cell 10
joblib.dump(pca,      "scania_pca.pkl")

print("SUCCESS: 'scania_rf_best.pkl', 'scania_scaler.pkl', and 'scania_pca.pkl' saved successfully!")

Fitting and saving production model and pipeline artifacts...
SUCCESS: 'scania_rf_best.pkl', 'scania_scaler.pkl', and 'scania_pca.pkl' saved successfully!
